# W3 Lab — Turning Functions into Tools

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ralbu85/stml_2026/blob/main/lectures/week03/W3_lab_tools.ipynb)

**Goal.** Turn a plain Python function into a tool the model can call, read the
request–execute–reinject trace behind each call, and show — with a measured routing
score — that tool choice is only as good as tool documentation. The closing email
assistant's capability is set entirely by its tool list.

Why this week: notes Ch. 3 set the division of labor — judgment to the model as text,
execution to code; this lab runs it live.

The path: setup → a model without tools → a first tool and its trace → a toolbox →
documentation as the interface → the routing score → the email assistant.

A **tool** = a function the model can request by name, with arguments, through the
function-calling protocol (notes Ch. 3).

> Adapted from DeepLearning.AI, *Agentic AI* by Andrew Ng — Module 3 (tool use):
> Sections 3–4 follow it closely, Sections 5–7 add the measured task and email
> assistant. Setup adapted for Colab and the course API standard.

*Runtime:* Google Colab, top-to-bottom, ~80 minutes. Cells marked ✍️ ask for your own
writing — a fill-in or a written prediction.


## 1. Setup

### 1.1 Installation

`aisuite` exposes multiple providers behind one interface and, given plain functions
in `tools=`, runs the full function-calling cycle itself. `qrcode` serves a Section 4
tool.

*Do:* run the setup cells in order — nothing to edit until the key cell.


In [ ]:
%pip install -q "aisuite[openai,anthropic]" "qrcode[pil]"

### 1.2 API key and model

Paste your key over `PASTE-YOUR-KEY-HERE` (issuing steps: the API Setup Guide on the
course site). The key bills to your account; keep the notebook private.


In [ ]:
import os

os.environ["OPENAI_API_KEY"] = "PASTE-YOUR-KEY-HERE"

MODEL = "openai:gpt-4o-mini"          # Anthropic accounts: "anthropic:claude-haiku-4-5"

### 1.3 Client and trace helper

`print_trace` prints the steps `aisuite` records for each tool call: tool requested,
arguments, return value, final message. Reading this trace is how every result in
this lab is verified.


In [ ]:
import aisuite

client = aisuite.Client()

def print_trace(response):
    """Response -> printed tool-call steps and final assistant message."""
    choice = response.choices[0]
    steps = getattr(choice, "intermediate_messages", None) or []
    for step in steps:
        calls = getattr(step, "tool_calls", None)
        if calls:
            for call in calls:
                print(f"[model -> tool] {call.function.name}({call.function.arguments})")
        elif isinstance(step, dict) and step.get("role") == "tool":
            print(f"[tool -> model] {step.get('name')}: {str(step.get('content'))[:200]}")
    if not steps:
        print("[no intermediate tool steps recorded]")
    print(f"[final] {choice.message.content}")

def called_tools(response):
    """Response -> list of tool names the model requested, in call order."""
    names = []
    for step in getattr(response.choices[0], "intermediate_messages", None) or []:
        for call in getattr(step, "tool_calls", None) or []:
            names.append(call.function.name)
    return names

### 1.4 Verification

In [ ]:
r = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Reply with exactly: ready"}],
)
print(r.choices[0].message.content)

Any error here is a setup problem, not a code problem — recheck the pasted key first.


## 2. A Model Without Tools

A model call returns text and nothing else — it cannot read the clock of the machine
it runs on (notes Ch. 1). The question below has a correct answer one line of Python
away.

*Do:* run the cell; keep the refusal for contrast with Section 3.


In [ ]:
r = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "What time is it?"}],
)
print(r.choices[0].message.content)

Every section below exists to close this gap.


## 3. A First Tool

### 3.1 The function

The tool is an ordinary function. `aisuite` reads its docstring to generate the JSON
schema the model sees (notes Ch. 3 writes that schema out by hand; here it is
derived).


In [ ]:
from datetime import datetime

def get_current_time():
    """Returns the current time as a string."""
    return datetime.now().strftime("%H:%M:%S")

get_current_time()

### 3.2 Passing the function as a tool

`tools=` receives the function object itself. `max_turns` caps the model↔tool round
trips — the guard against a model that keeps calling tools.


In [ ]:
messages = [{"role": "user", "content": "What time is it?"}]

response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=[get_current_time],
    max_turns=5,
)
print(response.choices[0].message.content)

The final message hides several intermediate steps, inspected next.

### 3.3 The trace


In [ ]:
print_trace(response)

Behind this trace `aisuite` performed the parse–execute–reinject steps; the protocol
is the one notes Ch. 3 walks through in JSON.

### 3.4 Manual schema and manual execution

The model actually receives a schema — `name`, `description`, `parameters`; writing
it by hand and handling the call yourself shows there is no other machinery.


In [ ]:
import json

tools_schema = [{
    "type": "function",
    "function": {
        "name": "get_current_time",
        "description": "Returns the current time as a string.",
        "parameters": {},
    },
}]

messages = [{"role": "user", "content": "What time is it?"}]

# With a hand-written schema, aisuite does not execute anything: no max_turns,
# and the tool call comes back for this code to handle.
response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=tools_schema,
)

tool_calls = response.choices[0].message.tool_calls
if tool_calls:
    call = tool_calls[0]
    result = get_current_time()

    messages.append(response.choices[0].message)
    messages.append({"role": "tool", "tool_call_id": call.id, "content": str(result)})

    final = client.chat.completions.create(model=MODEL, messages=messages, tools=tools_schema)
    print("requested:", call.function.name, call.function.arguments)
    print("final    :", final.choices[0].message.content)
else:
    print("model answered directly, no tool call requested:")
    print(response.choices[0].message.content)

The `tools=[function]` form of Section 3.2 automates exactly these steps and nothing
more.


## 4. A Toolbox

### 4.1 Tool definitions

Three more tools from the source lab: current weather (two keyless public APIs), a
text-file writer, a QR-code generator. With several tools registered, the model's job
now includes selection.


In [ ]:
import requests
import qrcode

def get_weather_from_ip():
    """Gets the current, high, and low temperature in Fahrenheit for the user's
    location and returns it as a string."""
    lat, lon = requests.get("https://ipinfo.io/json", timeout=10).json()["loc"].split(",")
    params = {
        "latitude": lat, "longitude": lon,
        "current": "temperature_2m",
        "daily": "temperature_2m_max,temperature_2m_min",
        "temperature_unit": "fahrenheit", "timezone": "auto",
    }
    data = requests.get("https://api.open-meteo.com/v1/forecast", params=params, timeout=10).json()
    return (f"Current: {data['current']['temperature_2m']}F, "
            f"High: {data['daily']['temperature_2m_max'][0]}F, "
            f"Low: {data['daily']['temperature_2m_min'][0]}F")

def write_txt_file(file_path: str, content: str):
    """Write a string into a .txt file (overwrites if the file exists).

    Args:
        file_path: Destination path.
        content: Text to write.
    """
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(content)
    return file_path

def generate_qr_code(data: str, filename: str):
    """Generate a QR code image encoding the given data.

    Args:
        data: Text or URL to encode.
        filename: Name for the output PNG file, without extension.
    """
    img = qrcode.make(data)
    output_file = f"{filename}.png"
    img.save(output_file)
    return f"QR code saved as {output_file}"

TOOLBOX = [get_current_time, get_weather_from_ip, write_txt_file, generate_qr_code]

### 4.2 Selection

The request below matches exactly one tool; the trace shows whether the model picked
it and ignored the other three.


In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Can you get the weather for my location?"}],
    tools=TOOLBOX,
    max_turns=5,
)
print_trace(response)

### 4.3 Arguments inferred from the request

`write_txt_file` takes two parameters. Both values exist only inside the user's
sentence; the model must extract them into the schema's fields.


In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content":
        "Can you make a txt note for me called reminders.txt "
        "that reminds me to call Daniel tomorrow at 7PM?"}],
    tools=TOOLBOX,
    max_turns=5,
)
print_trace(response)

if os.path.exists("reminders.txt"):
    with open("reminders.txt") as f:
        print("\nfile contents:", f.read())
else:
    print("\nreminders.txt not created (no tool execution in this run)")

The file on disk is the ground truth: the tool ran with the model's arguments, not
merely a claim in the final message.

### 4.4 A sequence of tool calls

One request, two tools, an ordering constraint: the note's content is the weather, so
`get_weather_from_ip` must run before `write_txt_file` — though the request mentions
the note first.


In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content":
        "Please make a QR code that goes to https://www.deeplearning.ai, "
        "call it dl_qr_code. Also write me a txt note named weather_note.txt "
        "with the current weather."}],
    tools=TOOLBOX,
    max_turns=10,
)
print_trace(response)

In [ ]:
from IPython.display import Image, display

if os.path.exists("dl_qr_code.png"):
    display(Image("dl_qr_code.png"))
else:
    print("dl_qr_code.png not created (no tool execution in this run)")

A correct trace runs `get_weather_from_ip` first: nothing in the code specified the
order — the model resolved the data dependency from the tool descriptions and the
request.


## 5. Documentation as the Interface ✍️

### 5.1 Two badly documented tools

The docstring is all the model ever sees of a function — the body never reaches it.
The two tools below work when called, but their docstrings describe nothing, so the
model has no basis for selection.


In [ ]:
PAPER_CATALOG = {
    "react":            {"title": "ReAct: Synergizing Reasoning and Acting in Language Models",
                         "authors": "Yao et al.", "year": 2022},
    "chain-of-thought": {"title": "Chain-of-Thought Prompting Elicits Reasoning in LLMs",
                         "authors": "Wei et al.", "year": 2022},
    "toolformer":       {"title": "Toolformer: Language Models Can Teach Themselves to Use Tools",
                         "authors": "Schick et al.", "year": 2023},
    "self-consistency": {"title": "Self-Consistency Improves Chain of Thought Reasoning",
                         "authors": "Wang et al.", "year": 2022},
}

def calculate(expression: str):
    """Does things with numbers."""
    allowed = set("0123456789+-*/(). ")
    if not set(expression) <= allowed:
        return "error: only arithmetic characters are allowed"
    return str(eval(expression))  # safe: input restricted to arithmetic characters

def paper_lookup(topic: str):
    """Looks stuff up."""
    entry = PAPER_CATALOG.get(topic.lower().strip())
    if entry is None:
        return f"error: no catalog entry for '{topic}'; known: {sorted(PAPER_CATALOG)}"
    return f"{entry['title']} — {entry['authors']}, {entry['year']}"

### 5.2 Selection with empty descriptions

Two requests these tools answer exactly. With no information in the descriptions, the
model picks the wrong tool, calls nothing, or guesses unaided.

*Do:* run the cell and note each misrouting.


In [ ]:
for question in ["What is 17 * 23?",
                 "In which year was the ReAct paper published, according to the course catalog?"]:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": question}],
        tools=[calculate, paper_lookup],
        max_turns=5,
    )
    print(f"--- {question}")
    print_trace(response)
    print()

### 5.3 Rewriting the docstrings ✍️

Rewrite both docstrings so the model can select and fill each tool reliably.
Requirements, matching the good-tool checklist (notes Ch. 3): one sentence stating
what the tool does and when to use it, and an `Args:` section describing each
parameter — its meaning and expected format (for `calculate`, state that the
expression uses Python arithmetic syntax; for `paper_lookup`, state the allowed topic
keys). Copy the shape of `write_txt_file`'s docstring in Section 4.1 — it satisfies
every requirement.

*Do:* fill in both docstrings here, then measure the effect in Section 6.3.


In [ ]:
### FILL IN (START) ###
CALCULATE_DOC = """"""

PAPER_LOOKUP_DOC = """"""
### FILL IN (END) ###

# Until the fill-ins are written, the vague originals stay in place.
if CALCULATE_DOC.strip():
    calculate.__doc__ = CALCULATE_DOC
if PAPER_LOOKUP_DOC.strip():
    paper_lookup.__doc__ = PAPER_LOOKUP_DOC

print(calculate.__doc__)
print(paper_lookup.__doc__)

## 6. Measured Task — Routing Score (core)

### 6.1 Task set

Eight requests: six answered through a specific tool, two directly with no tool call
— declining to call is also a routing decision. Each entry carries the expected tool
(or `None`) and a marker string a correct answer contains.


In [ ]:
TASKSET = [
    {"request": "What is 17 * 23?",
     "tool": "calculate", "marker": "391"},
    {"request": "What is (144 + 6) / 3?",
     "tool": "calculate", "marker": "50"},
    {"request": "In which year was the ReAct paper published, according to the course catalog?",
     "tool": "paper_lookup", "marker": "2022"},
    {"request": "Who are the authors of the Toolformer paper, according to the course catalog?",
     "tool": "paper_lookup", "marker": "Schick"},
    {"request": "Save a note file called todo.txt reminding me to submit the W4 lab.",
     "tool": "write_txt_file", "marker": "todo.txt"},
    {"request": "What time is it right now?",
     "tool": "get_current_time", "marker": ":"},
    {"request": "Define the term 'function calling' in one sentence.",
     "tool": None, "marker": "function"},
    {"request": "Is 'agent' a French word as well as an English one? Answer briefly.",
     "tool": None, "marker": "agent"},
]

ROUTING_TOOLS = [calculate, paper_lookup, write_txt_file, get_current_time]
TARGET_CORRECT = 6

### 6.2 Scorer

Correct = the expected tool appears in the trace, or — with no trace recorded — the
marker appears in the final answer. A no-tool task is wrong if any tool was called.


In [ ]:
#@title Scorer — run as-is (implements the rule described above) { display-mode: "form" }
def score_routing(taskset, tools):
    """Taskset -> number of correctly routed requests; prints one line per task."""
    correct = 0
    for task in taskset:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": task["request"]}],
            tools=tools,
            max_turns=5,
        )
        names = called_tools(response)
        text = (response.choices[0].message.content or "")
        if task["tool"] is None:
            ok = not names and task["marker"].lower() in text.lower()
        elif names:
            ok = task["tool"] in names
        else:
            ok = task["marker"].lower() in text.lower()
        correct += ok
        print(f"{'OK ' if ok else 'MISS'} expected={str(task['tool']):>16}  "
              f"called={names}  request={task['request'][:48]}")
    return correct

### 6.3 Run

Target: **at least 6 of 8** correct. Below target, the fix is in Section 5.3 — sharpen the two docstrings (state when to use the tool, name the parameter formats) and re-run from there. Docstring edits, not scorer edits, are the lever.

In [ ]:
routing_score = score_routing(TASKSET, ROUTING_TOOLS)
print(f"\nrouting score: {routing_score}/{len(TASKSET)}  (target: {TARGET_CORRECT})")

## 7. Email Assistant — Tools Define Capability

### 7.1 Simulated inbox

An agent for a stateful service, on the source lab's design, with the FastAPI backend
replaced by an in-memory list. The tools below are the complete action surface: the
agent can do exactly what they expose.


In [ ]:
INBOX_INITIAL = [
    {"id": 1, "sender": "boss@email.com",  "subject": "Quarterly report",
     "body": "Please send the Q3 numbers by Friday.", "read": False},
    {"id": 2, "sender": "alice@work.com",  "subject": "Happy Hour",
     "body": "Happy hour at 6pm on Thursday!", "read": True},
    {"id": 3, "sender": "newsletter@ml.org", "subject": "Weekly digest",
     "body": "This week in ML: new agent benchmarks.", "read": False},
]

INBOX = [dict(m) for m in INBOX_INITIAL]
SENT = []

def reset_inbox():
    """Restores the inbox to its initial three messages and clears sent mail."""
    INBOX[:] = [dict(m) for m in INBOX_INITIAL]
    SENT.clear()

def list_unread_emails():
    """Returns all unread emails as a list of dicts with id, sender, subject, body."""
    return [m for m in INBOX if not m["read"]]

def search_emails(query: str):
    """Returns emails whose sender, subject, or body contains the query string.

    Args:
        query: Case-insensitive substring to search for.
    """
    q = query.lower()
    return [m for m in INBOX
            if q in m["sender"].lower() or q in m["subject"].lower() or q in m["body"].lower()]

def mark_email_as_read(email_id: int):
    """Marks the email with the given id as read.

    Args:
        email_id: The id field of the email to mark.
    """
    for m in INBOX:
        if m["id"] == int(email_id):
            m["read"] = True
            return f"email {email_id} marked as read"
    return f"error: no email with id {email_id}"

def send_email(recipient: str, subject: str, body: str):
    """Sends an email.

    Args:
        recipient: Destination address.
        subject: Subject line.
        body: Message text.
    """
    SENT.append({"recipient": recipient, "subject": subject, "body": body})
    return f"sent to {recipient}: {subject}"

def delete_email(email_id: int):
    """Deletes the email with the given id.

    Args:
        email_id: The id field of the email to delete.
    """
    for m in INBOX:
        if m["id"] == int(email_id):
            INBOX.remove(m)
            return f"email {email_id} deleted"
    return f"error: no email with id {email_id}"

# Direct test of the tool layer, before any model is involved.
print(list_unread_emails())
print(search_emails("happy"))

### 7.2 Assistant preamble ✍️

Multi-step requests need standing instructions in front of the user's request. Write the preamble. Requirements:

1. state the assistant's role (email management);
2. grant use of the provided tools without asking for confirmation;
3. give the user's own address as `you@email.com` for outgoing mail.

*Do:* write the preamble in the next cell, then run it.

In [ ]:
### FILL IN (START) ###
EMAIL_PROMPT_PREAMBLE = """"""
### FILL IN (END) ###

def build_prompt(request: str) -> str:
    """User request -> preamble + request, as one prompt string."""
    return f"{EMAIL_PROMPT_PREAMBLE}\n\n{request.strip()}"

print(build_prompt("Delete the Happy Hour email"))

### 7.3 A multi-step request

Three actions in one sentence; the `SENT` list shows the side effect actually
happened.

*Do:* run the cell and read the trace — no code fixed the call order.


In [ ]:
EMAIL_TOOLS = [list_unread_emails, search_emails, mark_email_as_read, send_email]

reset_inbox()
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": build_prompt(
        "Check for unread emails from boss@email.com, mark them as read, "
        "and send a polite follow-up.")}],
    tools=EMAIL_TOOLS,
    max_turns=8,
)
print_trace(response)
print("\nSENT:", SENT)

### 7.4 Missing tool

The same agent, asked to delete an email — but `delete_email` is not in the tool list. The tool list is the capability boundary: no instruction can make the agent perform an action it has no tool for.

In [ ]:
reset_inbox()
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": build_prompt("Delete the Happy Hour email.")}],
    tools=EMAIL_TOOLS,           # delete_email deliberately absent
    max_turns=5,
)
print_trace(response)
print("\nHappy Hour still in inbox:", bool(search_emails("happy hour")))

Same request, `delete_email` now registered.

In [ ]:
reset_inbox()
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": build_prompt("Delete the Happy Hour email.")}],
    tools=EMAIL_TOOLS + [delete_email],
    max_turns=5,
)
print_trace(response)
print("\nHappy Hour still in inbox:", bool(search_emails("happy hour")))

Capability came from the tool list, not from the prompt — the prompt was identical in
both runs.


## 8. Completion Check

The cell reports each completion criterion. All rows must read `PASS` before submission; grading checks these same structural facts, never prose quality.

In [ ]:
completion = {
    "calculate docstring rewritten (>= 40 chars, has Args)":
        len(CALCULATE_DOC.strip()) >= 40 and "Args" in CALCULATE_DOC,
    "paper_lookup docstring rewritten (>= 40 chars, has Args)":
        len(PAPER_LOOKUP_DOC.strip()) >= 40 and "Args" in PAPER_LOOKUP_DOC,
    "email preamble written (>= 80 chars, mentions tools)":
        len(EMAIL_PROMPT_PREAMBLE.strip()) >= 80 and "tool" in EMAIL_PROMPT_PREAMBLE.lower(),
    "email preamble gives the user's address":
        "you@email.com" in EMAIL_PROMPT_PREAMBLE,
    f"routing score >= {TARGET_CORRECT}/8":
        routing_score >= TARGET_CORRECT,
}

for item, ok in completion.items():
    print(f"{'PASS' if ok else 'FAIL':4}  {item}")
print("\nLAB COMPLETE" if all(completion.values()) else "\nNOT COMPLETE YET")

---

Next week: the agent loop that carried this lab's multi-step traces, built by hand,
protocol and termination ours to set (notes Ch. 4). The sourced-report workflow that
closed the source module returns in the Week 7 project. Reference answers for lab and
homework: `labs/checkpoints/week03/solution.py`, published after the homework
deadline.
